# MODUL 14 — DEPLOYMENT DENGAN STREAMLIT & MINI PROYEK

**Nama:** Samuel Aldrik Sebastian  
**NIM:** 253307057  
**Mata Kuliah:** Sistem Cerdas

**Dosen Pengampu:** Muhammad Syaeful Fajar, S.Pd. Gr., M.Kom.

## Tujuan kegiatan
1. Menyimpan dan memuat model menggunakan `joblib`.
2. Membuat serta menjalankan aplikasi Streamlit yang menggunakan model machine learning.
3. Mengubah komponen antarmuka sesuai kebutuhan.
4. Mengembangkan kegiatan menggunakan metode berbeda untuk membantu masalah sehari-hari.

## Dataset yang digunakan
Sumber data: [kos_madiun.csv di GitHub](https://raw.githubusercontent.com/samuelsebastiann11-commits/Sistem-Cerdas/refs/heads/master/kos_madiun.csv)



## 1. Persiapan Google Colab

Library yang digunakan adalah Streamlit, joblib, scikit-learn, pandas, dan matplotlib. Instalasi dilakukan menggunakan `pip` karena notebook dijalankan di Google Colab.

In [ ]:
!pip -q install streamlit joblib scikit-learn pandas matplotlib
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null
!cloudflared --version

cloudflared version 2026.11.1 (built 2026-11-17-15:04 UTC)


**Catatan akses aplikasi:** Untuk menghindari kegagalan pemuatan komponen JavaScript Streamlit yang dapat terjadi saat melewati Cloudflare Tunnel, notebook ini menggunakan Cloudflare Quick Tunnel. Setiap aplikasi memakai port berbeda: 8501 untuk kegiatan utama, 8502 untuk kustomisasi, dan 8503 untuk pengembangan.

## 2. Memuat dan menyiapkan dataset

Dataset dibaca langsung dari GitHub sehingga tidak perlu mengunggah file secara manual. Sel ini juga memeriksa kolom, mengubah tipe data, membersihkan nilai yang tidak valid, dan mengubah kategori AC menjadi angka.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

URL_DATASET = "https://raw.githubusercontent.com/samuelsebastiann11-commits/Sistem-Cerdas/refs/heads/master/kos_madiun.csv"

df = pd.read_csv(URL_DATASET)
df.columns = df.columns.str.strip().str.lower()

kolom_wajib = ["jarak_km", "rating", "ac", "harga"]
kolom_hilang = [kolom for kolom in kolom_wajib if kolom not in df.columns]

if kolom_hilang:
    raise ValueError(f"Kolom wajib tidak ditemukan: {kolom_hilang}")

df["jarak_km"] = pd.to_numeric(df["jarak_km"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["harga"] = pd.to_numeric(df["harga"], errors="coerce")
df["ac"] = df["ac"].astype(str).str.strip().str.lower()
df = df[df["ac"].isin(["ya", "tidak"])].dropna(subset=kolom_wajib).copy()
df["ac_num"] = df["ac"].map({"ya": 1, "tidak": 0}).astype(int)

if len(df) < 5:
    raise ValueError("Jumlah data terlalu sedikit untuk pelatihan dan evaluasi model.")

print("Dataset berhasil dimuat dari GitHub.")
print(f"Jumlah data: {len(df)} baris")
print(f"Jumlah kolom: {len(df.columns)}")
display(df.head())
display(df.describe(include="all"))

Dataset berhasil dimuat dari GitHub.
Jumlah data: 30 baris
Jumlah kolom: 6


,nama,jarak_km,harga,ac,rating,ac_num
0,Kos Melati,1.8,600,tidak,4.3,0
1,Kos Mawar,0.6,1250,ya,3.7,1
2,Kos Anggrek,0.6,1200,ya,3.1,1
3,Kos Kenanga,2.3,500,ya,3.2,1
4,Kos Dahlia,2.3,1300,ya,4.9,1


,nama,jarak_km,harga,ac,rating,ac_num
count,30,30.000000,30.000000,30,30.000000,30.000000
unique,30,NaN,NaN,2,NaN,NaN
top,Kos Melati,NaN,NaN,tidak,NaN,NaN
freq,1,NaN,NaN,16,NaN,NaN
mean,NaN,2.320000,1056.666667,NaN,3.990000,0.466667
std,NaN,1.297584,279.078945,NaN,0.550454,0.507416
min,NaN,0.600000,500.000000,NaN,3.100000,0.000000
25%,NaN,1.350000,850.000000,NaN,3.700000,0.000000
50%,NaN,2.200000,1125.000000,NaN,4.050000,0.000000
75%,NaN,3.000000,1250.000000,NaN,4.375000,1.000000


# BAGIAN B — PRAKTIKUM TERBIMBING

## Kegiatan 1 — Instalasi dan Uji Streamlit

Pada modul, pengujian dilakukan dengan perintah `streamlit hello`. Di Google Colab, perintah demo dipanggil melalui proses Python. Setelah pengujian, proses demo dihentikan agar tidak mengganggu aplikasi yang akan dibuat.

In [ ]:
import subprocess
import time

uji_streamlit = subprocess.Popen(
    ["streamlit", "hello", "--server.headless", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(8)

if uji_streamlit.poll() is None:
    print("Demo Streamlit berhasil dijalankan.")
    uji_streamlit.terminate()
    uji_streamlit.wait(timeout=10)
    print("Proses demo dihentikan. Streamlit siap digunakan.")
else:
    raise RuntimeError("Demo Streamlit tidak berhasil dijalankan. Coba jalankan ulang sel instalasi.")

Demo Streamlit berhasil dijalankan.
Proses demo dihentikan. Streamlit siap digunakan.


## Kegiatan 2 — Menyimpan Model dengan `joblib`

Metode utama mengikuti kegiatan modul, yaitu **Linear Regression**. Variabel input yang digunakan adalah `jarak_km`, `rating`, dan `ac_num`, sedangkan target prediksi adalah `harga`.

In [ ]:
import joblib
from sklearn.linear_model import LinearRegression

fitur = ["jarak_km", "rating", "ac_num"]
X = df[fitur]
y = df["harga"]

model_linear = LinearRegression()
model_linear.fit(X, y)

joblib.dump(model_linear, "model_kos.joblib")

print("Model Linear Regression berhasil dilatih dan disimpan.")
print("Nama file: model_kos.joblib")
print("Koefisien:", model_linear.coef_.round(3))
print("Intercept:", round(model_linear.intercept_, 3))

Model Linear Regression berhasil dilatih dan disimpan.
Nama file: model_kos.joblib
Koefisien: [-28.531  27.07    7.373]
Intercept: 1011.41


### Penjelasan Kegiatan 2
- `ac_num` mengubah kategori fasilitas AC menjadi angka: `ya = 1` dan `tidak = 0`.
- `X` berisi variabel yang digunakan untuk memperkirakan harga kos.
- `y` berisi harga kos yang menjadi target prediksi.
- `LinearRegression()` membentuk model berdasarkan hubungan variabel input dengan harga.
- `joblib.dump()` menyimpan model ke file `model_kos.joblib` agar dapat digunakan kembali oleh aplikasi.

## Kegiatan 3 — Membuat dan Menjalankan Aplikasi `app_kos.py`

Aplikasi utama memuat model Linear Regression yang sudah disimpan. Pengguna dapat memasukkan jarak, rating, dan fasilitas AC untuk memperoleh perkiraan harga kos. Hasil harga dikonversi dari ribuan rupiah menjadi rupiah.

In [ ]:
app_kos = '\nimport streamlit as st\nimport pandas as pd\nimport joblib\n\nst.set_page_config(page_title="Prediksi Harga Kos Madiun", page_icon="🏠", layout="centered")\nst.title("Prediksi Harga Kos Madiun")\nst.write("Masukkan karakteristik kos untuk melihat perkiraan harga per bulan.")\n\n@st.cache_resource\ndef muat_model():\n    return joblib.load("model_kos.joblib")\n\nmodel = muat_model()\n\njarak = st.number_input("Jarak ke kampus (km)", min_value=0.0, value=2.0, step=0.1)\nrating = st.number_input("Rating kos", min_value=0.0, max_value=5.0, value=4.0, step=0.1)\nac = st.selectbox("Fasilitas AC", ["ya", "tidak"])\n\nif st.button("Prediksi Harga"):\n    data_baru = pd.DataFrame({\n        "jarak_km": [jarak],\n        "rating": [rating],\n        "ac_num": [1 if ac == "ya" else 0]\n    })\n    harga_ribu = float(model.predict(data_baru)[0])\n    harga_rupiah = harga_ribu * 1000\n    st.success(f"Perkiraan harga: Rp {harga_rupiah:,.0f} per bulan")\n    st.caption("Hasil merupakan perkiraan model, bukan harga resmi kos.")\n'

with open("app_kos.py", "w", encoding="utf-8") as file:
    file.write(app_kos)

print("File app_kos.py berhasil dibuat.")
print("File model_kos.joblib dan app_kos.py berada di folder kerja yang sama.")

File app_kos.py berhasil dibuat.
File model_kos.joblib dan app_kos.py berada di folder kerja yang sama.


### Menjalankan server aplikasi utama

Jalankan sel ini satu kali. Server berjalan pada port `8501`.

In [ ]:
import subprocess
import time
import os

if "proses_streamlit" in globals():
    if proses_streamlit.poll() is None:
        proses_streamlit.terminate()

log_streamlit = open("streamlit_main.log", "w")
proses_streamlit = subprocess.Popen(
    [
        "streamlit", "run", "app_kos.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log_streamlit,
    stderr=subprocess.STDOUT
)

time.sleep(8)

if proses_streamlit.poll() is not None:
    log_streamlit.close()
    print(open("streamlit_main.log", encoding="utf-8").read())
    raise RuntimeError("Server Streamlit gagal dijalankan.")

print("Server Streamlit aktif pada port 8501.")
print("Lanjutkan ke sel Cloudflare Tunnel untuk mendapatkan link publik.")

Server Streamlit aktif pada port 8501.
Lanjutkan ke sel Cloudflare Tunnel untuk mendapatkan link publik.


### Membuka aplikasi utama melalui Cloudflare Tunnel

Biarkan sel ini tetap berjalan. Buka alamat publik `https://....trycloudflare.com` yang muncul. Jika Cloudflare Tunnel menampilkan halaman pemeriksaan, ikuti instruksi pada halaman tersebut.

In [ ]:
import subprocess
import time
import re
from pathlib import Path
from IPython.display import display, Markdown

# Pastikan server Streamlit pada port ini sudah aktif.
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=5) as response:
        if response.status != 200:
            raise RuntimeError("Server Streamlit belum memberikan respons normal.")
except Exception as e:
    raise RuntimeError(
        "Server Streamlit belum aktif. Jalankan terlebih dahulu sel server untuk port 8501, "
        "lalu ulangi sel tunnel ini. Jangan membuka 127.0.0.1 di browser komputer."
    ) from e

# Hentikan tunnel lama jika sel dijalankan ulang.
if "tunnel_main" in globals() and tunnel_main.poll() is None:
    tunnel_main.terminate()
    try:
        tunnel_main.wait(timeout=5)
    except subprocess.TimeoutExpired:
        tunnel_main.kill()

log_path = "cloudflared_8501.log"
log_file = open(log_path, "w", encoding="utf-8")

tunnel_main = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8501"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print("Menghubungkan Cloudflare Tunnel...")
public_url = None
batas_tunggu = time.time() + 60

while time.time() < batas_tunggu:
    time.sleep(1)
    isi_log = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    cocok = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", isi_log)
    if cocok:
        public_url = cocok.group(0)
        break
    if tunnel_main.poll() is not None:
        break

if public_url:
    print("Tunnel berhasil dibuat.")
    display(Markdown(f"### Buka aplikasi: [{public_url}]({public_url})"))
    print("Biarkan sel server dan tunnel tetap berjalan selama aplikasi digunakan.")
else:
    isi_log = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    print("Alamat publik belum berhasil diperoleh.")
    print("Log Cloudflare (bagian akhir):")
    print(isi_log[-4000:] if isi_log else "Belum ada log.")
    raise RuntimeError(
        "Tunnel belum menghasilkan alamat trycloudflare.com. "
        "Pastikan runtime Colab terhubung ke internet, lalu jalankan ulang sel ini."
    )


Menghubungkan Cloudflare Tunnel...
Tunnel berhasil dibuat.


### Buka aplikasi: [https://src-dimensional-blues-fallen.trycloudflare.com](https://src-dimensional-blues-fallen.trycloudflare.com)

Biarkan sel server dan tunnel tetap berjalan selama aplikasi digunakan.


**Pengujian aplikasi utama**
1. Buka alamat publik `trycloudflare.com` yang muncul.
2. Masukkan jarak kos ke kampus.
3. Masukkan rating kos.
4. Pilih fasilitas AC.
5. Tekan **Prediksi Harga**.
6. Pastikan hasil perkiraan harga tampil.

Untuk menghentikan aplikasi, hentikan sel Cloudflare Tunnel dan proses server. Runtime Colab juga harus tetap aktif selama aplikasi digunakan.

## Kegiatan 4 — Kustomisasi Aplikasi

Kustomisasi dilakukan dengan mengganti judul aplikasi, menambahkan identitas pembuat, menyediakan pilihan kos dari dataset, dan menampilkan grafik rata-rata harga berdasarkan fasilitas AC. Input prediksi tetap menggunakan `jarak_km`, `rating`, dan `ac_num` karena hanya variabel tersebut yang digunakan saat model dilatih.

In [ ]:
app_kos_kustom = '\nimport streamlit as st\nimport pandas as pd\nimport joblib\n\nst.set_page_config(page_title="Aplikasi Perkiraan Biaya Kos", page_icon="🏡", layout="wide")\nst.title("Aplikasi Perkiraan Biaya Kos")\nst.write("Dibuat oleh: Samuel Aldrik Sebastian")\nst.write("Aplikasi untuk membantu membandingkan perkiraan biaya kos di Madiun.")\n\n@st.cache_resource\ndef muat_model():\n    return joblib.load("model_kos.joblib")\n\n@st.cache_data\ndef muat_data():\n    url = "https://raw.githubusercontent.com/samuelsebastiann11-commits/Sistem-Cerdas/refs/heads/master/kos_madiun.csv"\n    data = pd.read_csv(url)\n    data.columns = data.columns.str.strip().str.lower()\n    for kolom in ["jarak_km", "rating", "harga"]:\n        data[kolom] = pd.to_numeric(data[kolom], errors="coerce")\n    data["ac"] = data["ac"].astype(str).str.strip().str.lower()\n    return data.dropna(subset=["jarak_km", "rating", "harga", "ac"])\n\nmodel = muat_model()\ndata = muat_data()\n\nst.subheader("Data Kos Madiun")\nst.dataframe(data, use_container_width=True)\n\npilihan = st.selectbox("Pilih data kos sebagai referensi", data.index.tolist(),\n                       format_func=lambda i: str(data.loc[i, "nama"]) if "nama" in data.columns else f"Data kos {i+1}")\n\nbaris = data.loc[pilihan]\ncol1, col2 = st.columns(2)\nwith col1:\n    st.metric("Jarak ke kampus", f"{baris[\'jarak_km\']:.1f} km")\n    st.metric("Rating", f"{baris[\'rating\']:.1f} / 5")\nwith col2:\n    st.metric("Fasilitas AC", str(baris["ac"]).upper())\n    st.metric("Harga pada dataset", f"Rp {baris[\'harga\'] * 1000:,.0f}")\n\nst.subheader("Coba Prediksi dengan Kondisi Pilihan")\njarak = st.number_input("Jarak ke kampus (km)", min_value=0.0, value=float(baris["jarak_km"]), step=0.1)\nrating = st.number_input("Rating kos", min_value=0.0, max_value=5.0, value=float(baris["rating"]), step=0.1)\nac = st.selectbox("Fasilitas AC", ["ya", "tidak"], index=0 if baris["ac"] == "ya" else 1)\n\nif st.button("Hitung Perkiraan Biaya"):\n    data_baru = pd.DataFrame({\n        "jarak_km": [jarak],\n        "rating": [rating],\n        "ac_num": [1 if ac == "ya" else 0]\n    })\n    hasil = float(model.predict(data_baru)[0])\n    st.success(f"Perkiraan biaya: Rp {hasil * 1000:,.0f} per bulan")\n\nst.subheader("Rata-rata Harga Berdasarkan Fasilitas AC")\nrata_ac = data.groupby("ac")["harga"].mean().reindex(["ya", "tidak"]).dropna()\nrata_ac.index = ["Dengan AC" if x == "ya" else "Tanpa AC" for x in rata_ac.index]\nst.bar_chart(rata_ac * 1000)\nst.caption("Grafik menampilkan rata-rata harga berdasarkan data pada dataset.")\n'

with open("app_kos_kustom.py", "w", encoding="utf-8") as file:
    file.write(app_kos_kustom)

print("File app_kos_kustom.py berhasil dibuat.")

File app_kos_kustom.py berhasil dibuat.


### Menjalankan aplikasi kustom

Aplikasi kustom dijalankan pada port `8502` agar tidak bertabrakan dengan aplikasi utama. Jika aplikasi utama masih berjalan, aplikasi kustom tetap dapat dijalankan pada port yang berbeda.

In [ ]:
if "proses_streamlit_kustom" in globals():
    if proses_streamlit_kustom.poll() is None:
        proses_streamlit_kustom.terminate()

log_streamlit_kustom = open("streamlit_kustom.log", "w")
proses_streamlit_kustom = subprocess.Popen(
    [
        "streamlit", "run", "app_kos_kustom.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log_streamlit_kustom,
    stderr=subprocess.STDOUT
)

time.sleep(8)

if proses_streamlit_kustom.poll() is not None:
    log_streamlit_kustom.close()
    print(open("streamlit_kustom.log", encoding="utf-8").read())
    raise RuntimeError("Server aplikasi kustom gagal dijalankan.")

print("Aplikasi kustom aktif pada port 8502.")

Aplikasi kustom aktif pada port 8502.


### Link publik aplikasi kustom

Jalankan sel berikut dan buka link `loca.lt` yang ditampilkan.

In [ ]:
import subprocess
import time
import re
from pathlib import Path
from IPython.display import display, Markdown

# Pastikan server Streamlit pada port ini sudah aktif.
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:8502/_stcore/health", timeout=5) as response:
        if response.status != 200:
            raise RuntimeError("Server Streamlit belum memberikan respons normal.")
except Exception as e:
    raise RuntimeError(
        "Server Streamlit belum aktif. Jalankan terlebih dahulu sel server untuk port 8502, "
        "lalu ulangi sel tunnel ini. Jangan membuka 127.0.0.1 di browser komputer."
    ) from e

# Hentikan tunnel lama jika sel dijalankan ulang.
if "tunnel_kustom" in globals() and tunnel_kustom.poll() is None:
    tunnel_kustom.terminate()
    try:
        tunnel_kustom.wait(timeout=5)
    except subprocess.TimeoutExpired:
        tunnel_kustom.kill()

log_path = "cloudflared_8502.log"
log_file = open(log_path, "w", encoding="utf-8")

tunnel_kustom = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8502"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print("Menghubungkan Cloudflare Tunnel...")
public_url = None
batas_tunggu = time.time() + 60

while time.time() < batas_tunggu:
    time.sleep(1)
    isi_log = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    cocok = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", isi_log)
    if cocok:
        public_url = cocok.group(0)
        break
    if tunnel_kustom.poll() is not None:
        break

if public_url:
    print("Tunnel berhasil dibuat.")
    display(Markdown(f"### Buka aplikasi: [{public_url}]({public_url})"))
    print("Biarkan sel server dan tunnel tetap berjalan selama aplikasi digunakan.")
else:
    isi_log = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    print("Alamat publik belum berhasil diperoleh.")
    print("Log Cloudflare (bagian akhir):")
    print(isi_log[-4000:] if isi_log else "Belum ada log.")
    raise RuntimeError(
        "Tunnel belum menghasilkan alamat trycloudflare.com. "
        "Pastikan runtime Colab terhubung ke internet, lalu jalankan ulang sel ini."
    )


Menghubungkan Cloudflare Tunnel...
Tunnel berhasil dibuat.


### Buka aplikasi: [https://stones-november-offer-velvet.trycloudflare.com](https://stones-november-offer-velvet.trycloudflare.com)

Biarkan sel server dan tunnel tetap berjalan selama aplikasi digunakan.


### Penjelasan kustomisasi
1. Judul diubah menjadi **Aplikasi Perkiraan Biaya Kos** agar tujuan aplikasi lebih jelas.
2. Identitas pembuat dan deskripsi singkat ditampilkan menggunakan `st.write()`.
3. Dataset ditampilkan langsung pada aplikasi agar pengguna dapat melihat data kos yang tersedia.
4. `st.selectbox()` digunakan untuk memilih salah satu baris data kos sebagai referensi.
5. Input jarak, rating, dan fasilitas AC diisi berdasarkan data kos yang dipilih dan masih dapat diubah.
6. Grafik batang menampilkan rata-rata harga kos berdasarkan fasilitas AC.
7. Fitur nama kos hanya menjadi informasi tampilan. Model tidak menggunakan nama kos sebagai variabel prediksi.

# BAGIAN C — PENGEMBANGAN KEGIATAN

## Pengembangan: Prediksi biaya kos menggunakan Random Forest Regressor

### Masalah sehari-hari
Orang yang sedang mencari tempat tinggal sering membandingkan beberapa kos berdasarkan jarak ke kampus, rating, dan fasilitas AC. Harga yang ditawarkan dapat berbeda-beda. Perkiraan biaya dapat membantu calon penyewa menyusun pilihan sebelum melakukan survei langsung.

### Metode yang digunakan
Pengembangan menggunakan **Random Forest Regressor**, berbeda dari Linear Regression pada kegiatan utama. Random Forest membentuk sejumlah pohon keputusan dan menggabungkan hasilnya untuk menghasilkan prediksi.

Variabel tetap mengikuti modul:
- `jarak_km`: jarak kos ke kampus dalam kilometer.
- `rating`: penilaian kos.
- `ac_num`: fasilitas AC (`1` untuk ya dan `0` untuk tidak).
- `harga`: target harga kos per bulan dalam ribuan rupiah.

### Perbedaan dari kegiatan utama
1. Metode diubah dari `LinearRegression` menjadi `RandomForestRegressor`.
2. Variabel input dan target tetap sama agar pengembangan masih berkaitan dengan materi modul.
3. Model disimpan dengan nama `model_kos_rf.joblib`, sehingga tidak menimpa model utama.
4. Aplikasi dibuat terpisah dengan nama `app_pengembangan.py`.
5. Model dievaluasi menggunakan MAE dan R² sebagai gambaran performa pada data uji. Karena jumlah dataset terbatas, hasil evaluasi perlu dipahami sebagai evaluasi awal.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

X_rf = df[["jarak_km", "rating", "ac_num"]]
y_rf = df["harga"]

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y_rf, test_size=0.25, random_state=42
)

model_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
model_rf.fit(X_train, y_train)

prediksi_test = model_rf.predict(X_test)

mae = mean_absolute_error(y_test, prediksi_test)
print(f"MAE: {mae:.2f} ribu rupiah")
if len(y_test) > 1:
    nilai_r2 = r2_score(y_test, prediksi_test)
    print(f"R²: {nilai_r2:.3f}")

joblib.dump(model_rf, "model_kos_rf.joblib")
print("Model Random Forest tersimpan sebagai model_kos_rf.joblib.")

MAE: 271.00 ribu rupiah
R²: -2.242
Model Random Forest tersimpan sebagai model_kos_rf.joblib.


### Menguji prediksi pengembangan

Contoh berikut memperkirakan harga kos dengan jarak 1,2 km, rating 4,5, dan fasilitas AC tersedia. Nilai masukan dapat diubah sesuai kebutuhan.

In [ ]:
contoh_kos = pd.DataFrame({
    "jarak_km": [1.2],
    "rating": [4.5],
    "ac_num": [1]
})

hasil_rf = float(model_rf.predict(contoh_kos)[0])
print(f"Perkiraan harga kos: Rp {hasil_rf * 1000:,.0f} per bulan")

Perkiraan harga kos: Rp 889,000 per bulan


### Membuat aplikasi Streamlit untuk metode Random Forest

In [ ]:
app_pengembangan = '\nimport streamlit as st\nimport pandas as pd\nimport joblib\n\nst.set_page_config(page_title="Prediksi Biaya Kos - Random Forest", page_icon="🌳")\nst.title("Prediksi Biaya Kos untuk Mahasiswa")\nst.write("Aplikasi pengembangan menggunakan metode Random Forest Regressor.")\nst.write("Masukkan jarak, rating, dan fasilitas kos untuk melihat perkiraan biaya.")\n\n@st.cache_resource\ndef muat_model():\n    return joblib.load("model_kos_rf.joblib")\n\nmodel = muat_model()\n\njarak = st.number_input("Jarak ke kampus (km)", min_value=0.0, value=2.0, step=0.1)\nrating = st.number_input("Rating kos", min_value=0.0, max_value=5.0, value=4.0, step=0.1)\nac = st.selectbox("Apakah tersedia AC?", ["ya", "tidak"])\n\nif st.button("Perkirakan Biaya Kos"):\n    data_baru = pd.DataFrame({\n        "jarak_km": [jarak],\n        "rating": [rating],\n        "ac_num": [1 if ac == "ya" else 0]\n    })\n    hasil = float(model.predict(data_baru)[0])\n    st.success(f"Perkiraan biaya: Rp {hasil * 1000:,.0f} per bulan")\n    st.caption("Hasil merupakan perkiraan berdasarkan data pelatihan, bukan harga resmi kos.")\n'

with open("app_pengembangan.py", "w", encoding="utf-8") as file:
    file.write(app_pengembangan)

print("File app_pengembangan.py berhasil dibuat.")

File app_pengembangan.py berhasil dibuat.


### Menjalankan aplikasi pengembangan

Server Random Forest menggunakan port `8503` agar terpisah dari aplikasi utama dan aplikasi kustom.

In [ ]:
if "proses_streamlit_rf" in globals():
    if proses_streamlit_rf.poll() is None:
        proses_streamlit_rf.terminate()

log_streamlit_rf = open("streamlit_rf.log", "w")
proses_streamlit_rf = subprocess.Popen(
    [
        "streamlit", "run", "app_pengembangan.py",
        "--server.port", "8503",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log_streamlit_rf,
    stderr=subprocess.STDOUT
)

time.sleep(8)

if proses_streamlit_rf.poll() is not None:
    log_streamlit_rf.close()
    print(open("streamlit_rf.log", encoding="utf-8").read())
    raise RuntimeError("Server aplikasi Random Forest gagal dijalankan.")

print("Aplikasi pengembangan aktif pada port 8503.")

Aplikasi pengembangan aktif pada port 8503.


### Membuka aplikasi pengembangan melalui Cloudflare Tunnel

Jalankan sel berikut, kemudian buka alamat publik `trycloudflare.com` yang muncul.

In [ ]:
import subprocess
import time
import re
from pathlib import Path
from IPython.display import display, Markdown

# Pastikan server Streamlit pada port ini sudah aktif.
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:8503/_stcore/health", timeout=5) as response:
        if response.status != 200:
            raise RuntimeError("Server Streamlit belum memberikan respons normal.")
except Exception as e:
    raise RuntimeError(
        "Server Streamlit belum aktif. Jalankan terlebih dahulu sel server untuk port 8503, "
        "lalu ulangi sel tunnel ini. Jangan membuka 127.0.0.1 di browser komputer."
    ) from e

# Hentikan tunnel lama jika sel dijalankan ulang.
if "tunnel_rf" in globals() and tunnel_rf.poll() is None:
    tunnel_rf.terminate()
    try:
        tunnel_rf.wait(timeout=5)
    except subprocess.TimeoutExpired:
        tunnel_rf.kill()

log_path = "cloudflared_8503.log"
log_file = open(log_path, "w", encoding="utf-8")

tunnel_rf = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8503"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print("Menghubungkan Cloudflare Tunnel...")
public_url = None
batas_tunggu = time.time() + 60

while time.time() < batas_tunggu:
    time.sleep(1)
    isi_log = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    cocok = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", isi_log)
    if cocok:
        public_url = cocok.group(0)
        break
    if tunnel_rf.poll() is not None:
        break

if public_url:
    print("Tunnel berhasil dibuat.")
    display(Markdown(f"### Buka aplikasi: [{public_url}]({public_url})"))
    print("Biarkan sel server dan tunnel tetap berjalan selama aplikasi digunakan.")
else:
    isi_log = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    print("Alamat publik belum berhasil diperoleh.")
    print("Log Cloudflare (bagian akhir):")
    print(isi_log[-4000:] if isi_log else "Belum ada log.")
    raise RuntimeError(
        "Tunnel belum menghasilkan alamat trycloudflare.com. "
        "Pastikan runtime Colab terhubung ke internet, lalu jalankan ulang sel ini."
    )


Menghubungkan Cloudflare Tunnel...
Tunnel berhasil dibuat.


### Buka aplikasi: [https://elvis-revisions-creativity-tuesday.trycloudflare.com](https://elvis-revisions-creativity-tuesday.trycloudflare.com)

Biarkan sel server dan tunnel tetap berjalan selama aplikasi digunakan.


### Pengujian hasil pengembangan
1. Buka link aplikasi Random Forest yang muncul dari Cloudflare Tunnel.
2. Masukkan jarak kos ke kampus dalam kilometer.
3. Masukkan rating kos antara 0 sampai 5.
4. Pilih fasilitas AC.
5. Tekan **Perkirakan Biaya Kos**.
6. Amati hasil perkiraan harga yang ditampilkan.

**Catatan:** nilai `harga` pada dataset dinyatakan dalam ribuan rupiah. Hasil model dikalikan 1.000 untuk ditampilkan dalam rupiah. Prediksi merupakan perkiraan berdasarkan data yang tersedia, bukan harga pasti.

# LATIHAN MANDIRI — JAWABAN

**1. Mengapa input baru tidak boleh langsung dimasukkan ke model jika model tidak dilatih dengan input tersebut?**  
Karena model hanya mengenali fitur yang digunakan saat pelatihan. Penambahan fitur yang tidak dikenal dapat menyebabkan ketidaksesuaian jumlah atau susunan kolom sehingga prediksi gagal atau tidak valid.

**2. Mengapa susunan kolom pada `data_baru` harus sama dengan saat model dilatih?**  
Karena setiap kolom mewakili fitur tertentu. Jika nama atau urutannya berbeda, model dapat membaca nilai pada fitur yang keliru sehingga hasil prediksi tidak sesuai.

**3. Sebutkan dua topik AI yang menarik dan alasannya.**  
- Machine Learning untuk prediksi, karena dapat membantu memperkirakan suatu nilai berdasarkan data sebelumnya.
- Sistem rekomendasi, karena dapat membantu pengguna memilih barang atau layanan sesuai kebutuhan.

**4. Satu kursus atau sertifikasi yang akan diikuti dalam enam bulan.**  
Mengikuti Kaggle Learn, khususnya materi *Intro to Machine Learning*, dengan target menyelesaikan materi dan membuat satu latihan model prediksi sederhana dalam enam bulan.

**5. Tiga langkah kecil yang akan dilakukan pada bulan depan.**  
- Minggu pertama: mempelajari kembali dasar Python dan pengolahan data.
- Minggu kedua: mengikuti materi awal Kaggle Learn.
- Minggu ketiga dan keempat: mencoba satu dataset sederhana, membuat model prediksi, lalu mencatat hasil dan evaluasinya.

# KESIMPULAN

Deployment menghubungkan model machine learning dengan pengguna melalui aplikasi Streamlit. Pada kegiatan utama, model Linear Regression disimpan menggunakan `joblib` dan digunakan untuk memperkirakan harga kos berdasarkan jarak, rating, serta fasilitas AC. Kustomisasi menambahkan tampilan data kos, pilihan data, dan grafik rata-rata harga. Pada bagian pengembangan, metode diganti menjadi Random Forest Regressor untuk menyelesaikan masalah yang sama dengan variabel yang tetap. Dengan demikian, perbedaan utama terletak pada metode pemodelan, sedangkan tujuan aplikasinya tetap membantu memperkirakan biaya kos.

**Catatan pengumpulan:** notebook ini menggunakan dataset dari GitHub. File `.py` dan `.joblib` dibuat otomatis saat sel terkait dijalankan. Link aplikasi hanya aktif selama runtime Colab dan proses server/tunnel berjalan.